#تمرین پنجم درس بینایی کامپیوتر
در این سوال سعی در مقایسه دو الگوریتم خوشه‌بندی مبتنی بر روش $Expectation Maximization$ داریم. برای این کار ابتدا یکسری نقاط را تولید کرده و در قدم اول از الگوریتم $KMeans$ برای خوشه‌بندی آنها استفاده می‌کنیم و نتایج را نمایش می‌دهیم. در قدم بعدی الگوریتم $Gussian Mixture Model$ را روی همان نقاط اجرا 
کرده و نتایج را به قسمت قبل مقایسه می‌کنیم

کتابخانه‌های لازم را در زیر اضافه کنید

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler


### تولید داده

در این بخش ابتدا یک تابع برای تولید داده‌ها مینویسم. این تابع با گرفتن تعداد خوشه، و نیز میانگین و واریانس برای هر خوشه، هرکدام را یک توزیع گاوسی با میانگین و واریانس داده شده در نظر میگیرد و با نمونه برداری تعدادی نقطه مشخص از هرکدام، داده‌ها را تولید می‌کند.

In [ ]:
def data_generator(cluster_nums, cluster_means, cluster_var,
                   background_range, background_noise_nums):
    data = []
    for num, mu, var in zip(cluster_nums, cluster_means, cluster_var):
        data += [np.random.multivariate_normal(mu, np.diag(var), num)]
    data = np.vstack(data)
    noise = np.random.uniform(background_range[0], background_range[1], size=(background_noise_nums, data.shape[-1]))
    data = np.append(data, noise, axis=0)
    return data

در قطعه کد زیر با فراخوانی تابع بالا، یکسری داده از سه توزیع گاوسی تولید میکنیم و آنها را نمایش می‌دهیم. دقت شود که ابتدا نقاط را نرمال می‌کنیم. علت این کار این است که همه آنها در یک $Scale$ باشند. از آنجا که روش‌های خوشه بندی مانند $KMeans$ از فاصله بین نقاط استفاده می‌کنند، لذا نرمال سازی برای بهبود و نتایج لازم است

In [ ]:
X = data_generator(cluster_nums=[400,600,800],
                   cluster_means=[[0.5, 0.5],
                                  [6, 1.5],
                                  [1, 7]],
                   cluster_var=[[1, 3],
                                [2, 2],
                                [6, 2]],
                   background_range=[[-10, -15],
                                     [15, 20]],
                   background_noise_nums=30)



### K-Means Clustering
الگوریتم $kmeans$ یک الگوریتم تکرار شونده است که سعی در خوشه‌بندی داده‌ها به صورت بدون نظارت در تعدادی دسته از پیش مشخص شده مانند $K$ را دارد

مراحل انجام این الگوریتم:
- ابتدا تعداد دسته‌ها یا همان $K$ مشخص می‌شود
- سپس به ازای هر دسته معمولا به صورت تصادفی یک نقطه به عنوان مرکز آن در نظر گرفته می‌شود
- برای هر نقطه، فاصله اقلیدوسی آن تا مرکز هر یک از خوشه‌ها محاسبه می‌شود و خوشه با کمترین فاصله به عنوان خوشه داده انتخاب می‌شود
- با بدست آمدن خوشه هر داده، این بار مرکز هر خوشه آپدیت می‌شود، به این صورت که میانگین داده‌های هر خوشه به عنوان مرکز جدید خوشه در نظر گرفته می‌شود

این نوع کارکرد را معمولا تحت عنوان الگوریتم‌های $EM$ می‌شناسیم. این الگوریتم‌ها از دو قسمت تخمین $E$ و یک بخش بیشینه‌سازی $M$ تشکیل می‌شوند. در قسمت تخمین به هر داده یک خوشه نسبت داده می‌شود و در قسمت بیشینه سازی مرکز جدید هر خوشه محسابه می‌شود


به صورت رسمی تر، تابع هزینه این الگوریتم به صورت زیر است<br>
$$
    \large J = \sum_{i=1}^{m} \sum_{k=1}^{K} \mathbb{I}(z_i = k)||x_i - \mu_k||_2^2
$$<br>

که در آن $z_i$ خوشه نسبت داده شده به داده $x_i$ و $\mu_k$ نیز میانگین خوشه شماره $k$ است

حال گام $E$ به صورت زیر است<br>
$$
    \large z_i^{*} = \text{argmin}_{k} ||x_i - \mu_k||_2^2
$$<br>

و گام $M$ نیز به صورت زیر<br>
$$
    \large \mu_k = \frac{1}{\sum_{i=1}^m \mathbb{I}(z_i = k)} \sum_{i=1}^m  x_i\mathbb{I}(z_i = k)
$$<br>

در عمل از آنجا که نتایج نهایی الگوریتم وابسته به مقدار دهی اولیه مراکز خوشه‌ها است، الگوریتم را چندین بار با مراکز اولیه متفاوت اجرا می‌کنند و در نهایت بهترین نتیجه بدست آمده جواب نهایی الگوریتم خواهد بود

در قسمت زیر یک نمونه اجرای الگوریتم kmeans بر روی داده‌های تولید شده را تست می‌کنیم. همانطور که مشخص است در ورودی الگوریتم تعداد خوشه‌ها از قبل مشخص می‌شود و همچنین تعداد تکرار الگوریتم نیز می‌‌تواند به عنوان ورودی به تابع داده شود. در نهایت نتایج رسم شده است و می‌توان دید که در خروجی این الگوریتم اشتباهاتی وجود دارد و خوشه‌ها کاملا خوب مشخص نشده اند. در واقع مدل نتوانسته همان توزیع‌های گاوسی که داده‌ها را از آنها نمونه برداری کرده ایم را بدست بیاورد

### Gaussian Mixture Model

در این روش هر خوشه را یک توزیع نرمال در نظر می‌گیریم و سعی می‌کنیم با یک روش تکرار شونده هربار احتمال تعلق هر داده را به هر یک از خوشه‌ها حساب کنیم و در نهایت طی هر تکرار پارامترهای توزیع نرمال را تخمین زده و بهبود دهیم

فرض کنید که یک مدل مخلوط گاوسی با $K$ توزیع نرمال داریم. $X$  متغییر دیده شده و $Z$ متغییر نهان. حال داریم

$$
\large
\begin{aligned}
    P(\boldsymbol{x}_i | \boldsymbol{\theta}) &= \sum_{k=1}^{K} \pi_k P(\boldsymbol{x}_i | \boldsymbol{z}_i, \boldsymbol{\theta}_k)\\
    &= \sum_{k=1}^{K} P(\boldsymbol{z}_i = k) \mathcal{N}(\boldsymbol{x}_i; \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k)
\end{aligned}
$$<br>

در قدم $E$ الگوریتم، احتمال تعلق داده‌ها را به هر خوشه حساب می‌کنیم. در قدم $M$ اما پارامترهای توزیع‌های نرمال را بر اساس تخمین‌های زده شده در قسمت قبل دوباره حساب می‌کنیم. درواقع در گام $E$ محاسبات به صورت زیر است

$$
\large
    f(\boldsymbol{x_i}|\mu_k, \boldsymbol{\Sigma}_k) = \frac{1}{\sqrt{(2\pi)^m|\boldsymbol{\Sigma}_k|}} \exp\left(-\frac{(\boldsymbol{x}_i - \boldsymbol{\mu}_k)^\top \boldsymbol{\Sigma}_k^{-1}(\boldsymbol{x}_i - \boldsymbol{\mu}_k)}{2}\right)
$$<br>

سپس احتمال تعلق داده $x_i$ به توزیع $k$ ام به صورت زیر محاسبه می‌شود

$$
\Large
    p_{ik} = \frac{\pi_k f(\boldsymbol{x_i}|\mu_k, \boldsymbol{\Sigma}_k)}{\sum_{j=1}^{K} \pi_j f(\boldsymbol{x_i}|\mu_j, \boldsymbol{\Sigma}_j)}
$$<br>

در گام $M$ اما پارامترهای توزیع‌ها طبق تخمین‌های قسمت قبل دوباره محاسبه می‌شوند
$$
\Large
\begin{aligned}
    \pi_k &= \frac{1}{N}\sum_{i=1}^{N} p_{ik}\\
    \boldsymbol{\mu}_k &= \frac{1}{\sum_{i=1}^{N} p_{ik}} \sum_{i=1}^{N} p_{ik} \boldsymbol{x_i}\\
    \boldsymbol{\Sigma}_k &= \frac{1}{\sum_{i=1}^{N} p_{ik}} \sum_{i=1}^{N} p_{ik} (\boldsymbol{x}_i - \boldsymbol{\mu}_k) (\boldsymbol{x}_i - \boldsymbol{\mu}_k)^\top
\end{aligned}
$$

در قطعه کد زیر یک مدل $GMM$ برای سه خوشه بر روی داده‌ها تولید شده در قسمت اول تمرین آماده شده و خوشه هر داده مشخص می‌شود. نتایج حاصل نیز نمایش داده شده است

### نتیجه گیری 
تعبیر خود را از مقایسه دو الگوریتم اجرا شده را بنویسید